# Deliverable 1: Embeddings Evolution & Semantic Search
Comparing Word2Vec, GloVe, and SBERT, and evaluating **Cosine Similarity vs. Dot Product** for semantic search.

In [1]:
import numpy as np
import pandas as pd
import gensim.downloader as api
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries loaded successfully!")

Libraries loaded successfully!


## 1. Load Embedding Models
We load models across three generations of embeddings:
1. **Word2Vec:** Static local context windows (~1.66 GB download on first run).
2. **GloVe:** Static global co-occurrence frequencies (~66 MB download).
3. **SBERT:** Contextual sentence embeddings via Transformer architecture.

In [2]:
# 1. Word2Vec (Google News, 300 dimensions)
# Optional: Uncomment below if you want to download and test the full 1.66 GB Google News model
# print("Loading Word2Vec model (~1.66 GB download on first run)...\n")
# w2v_model = api.load("word2vec-google-news-300")
# print("Word2Vec loaded!")

In [3]:
# 2. GloVe (Wikipedia 6B tokens, 100 dimensions, ~66 MB download)
print("Loading GloVe model (~128 MB uncompressed)...\n")
glove_model = api.load("glove-wiki-gigaword-100")
print("GloVe loaded successfully!")

Loading GloVe model (~128 MB uncompressed)...

GloVe loaded successfully!


In [4]:
# 3. Sentence Transformers (all-MiniLM-L6-v2, 384 dimensions)
print("Loading SBERT model...\n")
sbert_model = SentenceTransformer("all-MiniLM-L6-v2")
print("SBERT loaded successfully!")

Loading SBERT model...



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

SBERT loaded successfully!


## 2. Sentence Pooling for Word Embeddings
Because Word2Vec and GloVe produce embeddings for single words rather than whole sentences, we compute the document vector by **averaging the embeddings** of all recognized words.

In [5]:
def get_average_embedding(text: str, model) -> np.ndarray:
    words = text.lower().split()
    # Collect vectors for words present in the model vocabulary
    vectors = [model[word] for word in words if word in model]
    
    if not vectors:
        return np.zeros(model.vector_size)
    
    return np.mean(vectors, axis=0)

## 3. Similarity Functions: Cosine Similarity vs. Dot Product
* **Cosine Similarity:** Measures the angle between vectors, normalizing for vector magnitude.
* **Dot Product:** Measures directional alignment multiplied by vector length.

In [6]:
def cosine_sim(u: np.ndarray, v: np.ndarray) -> float:
    norm_u = np.linalg.norm(u)
    norm_v = np.linalg.norm(v)

    # Handle zero vectors to prevent division by zero
    if norm_u == 0 or norm_v == 0:
        return 0.0
    return float(np.dot(u, v) / (norm_u * norm_v))

def dot_sim(u: np.ndarray, v: np.ndarray) -> float:
    return float(np.dot(u, v))

## 4. Test Sentences: Breaking the Bag-of-Words Paradigm
We test three sentences:
* `sent_a`: "The dog bit the man" (Active voice)
* `sent_b`: "The man bit the dog" (Same words, inverted syntax/meaning)
* `sent_c`: A long paragraph discussing stocks, war, and peace negotiations (Completely different domain)

In [7]:
sent_a = "The dog bit the man"
sent_b = "The man bit the dog"
sent_c = "The stocks are plummeting, where the financial situation is getting worse due to the ongoing war. The people are demanding an immediate cease of fire, taking the conflict to the negotiation table in a desperate hope for peace."

### Evaluation: Word2Vec (Optional)

In [8]:
# Evaluates Word2Vec only if loaded into memory
if 'w2v_model' in globals():
    w2v_a = get_average_embedding(sent_a, w2v_model)
    w2v_b = get_average_embedding(sent_b, w2v_model)
    w2v_c = get_average_embedding(sent_c, w2v_model)
    print("Word2Vec Cosine (A vs B):", cosine_sim(w2v_a, w2v_b))
    print("Word2Vec Cosine (A vs C):", cosine_sim(w2v_a, w2v_c))
else:
    print("Word2Vec skipped (w2v_model not loaded). Run cell #1 to load it if desired.")

Word2Vec skipped (w2v_model not loaded). Run cell #1 to load it if desired.


### Evaluation: GloVe (Static Word Vectors + Mean Pooling)
Notice how GloVe processes individual words and averages them across the sentence.

In [9]:
glove_a = get_average_embedding(sent_a, glove_model)
glove_b = get_average_embedding(sent_b, glove_model)
glove_c = get_average_embedding(sent_c, glove_model)

print("=== GloVe Results ===")
print(f"Cosine Sim  (A vs B - Order swap):    {cosine_sim(glove_a, glove_b):.4f}")
print(f"Cosine Sim  (A vs C - War/Stocks):     {cosine_sim(glove_a, glove_c):.4f}")
print(f"Dot Product (A vs B - Order swap):    {dot_sim(glove_a, glove_b):.4f}")
print(f"Dot Product (A vs C - War/Stocks):     {dot_sim(glove_a, glove_c):.4f}")

=== GloVe Results ===
Cosine Sim  (A vs B - Order swap):    1.0000
Cosine Sim  (A vs C - War/Stocks):     0.8674
Dot Product (A vs B - Order swap):    20.6072
Dot Product (A vs C - War/Stocks):     17.2014


### Evaluation: SBERT (Contextual Transformer Embeddings)
Notice how SBERT processes the sentence as a unified sequence with self-attention.

In [10]:
sbert_a = sbert_model.encode(sent_a)
sbert_b = sbert_model.encode(sent_b)
sbert_c = sbert_model.encode(sent_c)

print("=== SBERT Results ===")
print(f"Cosine Sim  (A vs B - Order swap):    {cosine_sim(sbert_a, sbert_b):.4f}")
print(f"Cosine Sim  (A vs C - War/Stocks):     {cosine_sim(sbert_a, sbert_c):.4f}")
print(f"Dot Product (A vs B - Order swap):    {dot_sim(sbert_a, sbert_b):.4f}")
print(f"Dot Product (A vs C - War/Stocks):     {dot_sim(sbert_a, sbert_c):.4f}")

=== SBERT Results ===
Cosine Sim  (A vs B - Order swap):    0.9795
Cosine Sim  (A vs C - War/Stocks):     0.0647
Dot Product (A vs B - Order swap):    0.9795
Dot Product (A vs C - War/Stocks):     0.0647


## 5. Direct Side-by-Side Comparison Matrix

In [11]:
results = [
    {
        "Sentence Comparison": "A vs B (Dog bit man vs Man bit dog)",
        "GloVe Cosine": cosine_sim(glove_a, glove_b),
        "GloVe Dot": dot_sim(glove_a, glove_b),
        "SBERT Cosine": cosine_sim(sbert_a, sbert_b),
        "SBERT Dot": dot_sim(sbert_a, sbert_b)
    },
    {
        "Sentence Comparison": "A vs C (Dog bit man vs Stock market crash)",
        "GloVe Cosine": cosine_sim(glove_a, glove_c),
        "GloVe Dot": dot_sim(glove_a, glove_c),
        "SBERT Cosine": cosine_sim(sbert_a, sbert_c),
        "SBERT Dot": dot_sim(sbert_a, sbert_c)
    }
]

df_results = pd.DataFrame(results)
display(df_results.round(4))

,Sentence Comparison,GloVe Cosine,GloVe Dot,SBERT Cosine,SBERT Dot
0,A vs B (Dog bit man vs Man bit dog),1.0000,20.6072,0.9795,0.9795
1,A vs C (Dog bit man vs Stock market crash),0.8674,17.2014,0.0647,0.0647


## 6. Analysis & Findings: Why These Numbers Make Perfect Sense

### 1. GloVe gives EXACTLY 1.0000 on A vs. B (The Bag-of-Words Trap)
* `sent_a` ("The dog bit the man") and `sent_b` ("The man bit the dog") contain the exact same words.
* Because sentence pooling simply averages word vectors, vector addition is commutative: `A + B = B + A`.
* GloVe is completely blind to word order and syntax, rating them as 100% identical.
* **SBERT (0.9795):** Captures that both sentences share the same topic and scene, but acknowledges the syntactic difference through Transformer self-attention.

---

### 2. GloVe gives 0.8674 on A vs. C (Semantic Dilution / Anisotropy)
* Sentence C is a 33-word paragraph about financial collapse and war.
* When you average 30+ word vectors together alongside common stop words, specific semantic signals are diluted into a generic "average English language" vector.
* Word embeddings suffer from **anisotropy** (all word vectors tend to occupy a narrow cone in vector space), artificially inflating cosine similarities for long, unrelated texts.
* **SBERT (0.0647):** Correctly detects near-zero semantic relationship between a dog biting a man and economic collapse.

---

### 3. SBERT Cosine Similarity == Dot Product (0.9795 == 0.9795)
* Notice that for SBERT, Cosine Similarity and Dot Product produce the exact same numbers.
* This occurs because `SentenceTransformer` outputs $L_2$-normalized vectors where $\|u\| = 1$ and $\|v\| = 1$.
* When vectors are normalized to unit length, the cosine denominator simplifies: `(u · v) / (1 * 1) = u · v`.
* In GloVe, vectors are unnormalized (lengths are ~4.5), causing raw dot products to blow up to arbitrary values (20.6 and 17.2).

## 7. Unified Embedder Class (Adapter Pattern)
We create a single wrapper class `TextEmbedder` that accepts **any** model (Word2Vec, GloVe, or SBERT) and exposes a unified `.encode(texts)` method. This allows us to swap models without changing downstream retrieval logic.

In [12]:
class TextEmbedder:
    def __init__(self, model):
        self.model = model

    def encode(self, texts):
        # Handle both a single string or a list of strings
        is_single = isinstance(texts, str)
        text_list = [texts] if is_single else texts

        # If the model has its own .encode() method (e.g. SBERT)
        if hasattr(self.model, "encode"):
            embeddings = self.model.encode(text_list)
        else:
            # Otherwise, use mean-pooling for word vectors (GloVe / Word2Vec)
            embeddings = np.array([get_average_embedding(t, self.model) for t in text_list])

        return embeddings[0] if is_single else embeddings

## 8. Corpus Indexing
We define our document corpus and embed it using the unified TextEmbedder.

In [13]:
documents = [
    "Commercial banks manage deposits, loans, and corporate accounts.",
    "A cozy Italian restaurant serving handmade pasta and pizza.",
    "Central banks regulate interest rates and national inflation.",
    "A rich chocolate espresso cake garnished with strawberries.",
]

# Easily choose and swap your model here:
embedder = TextEmbedder(sbert_model)

# Embed the entire corpus once (like building an offline Vector DB index)
doc_embeddings = embedder.encode(documents)
print(f"Corpus indexed! Matrix shape: {doc_embeddings.shape}")

Corpus indexed! Matrix shape: (4, 384)


In [14]:
# Step A: The user types a query 
user_query = "Where can I eat something delicious?"

# Step B: Embed the search query using the SAME model
query_vec = embedder.encode(user_query)

# Step C: Score the query against all 4 documents
cos_scores = [cosine_sim(query_vec, doc_vec) for doc_vec in doc_embeddings]
dot_scores = [dot_sim(query_vec, doc_vec) for doc_vec in doc_embeddings]

# Step D: Rank ALL documents by relevance (highest to lowest score)
print("=== SBERT Search Results ===")
ranked_cos_idx = np.argsort(cos_scores)[::-1]
print("\n--- SBERT Cosine Similarity Rankings ---")
for rank, idx in enumerate(ranked_cos_idx, 1):
    print(f"Rank {rank} (Score: {cos_scores[idx]:.4f}): {documents[idx]}")

ranked_dot_idx = np.argsort(dot_scores)[::-1]
print("\n--- SBERT Dot Product Rankings ---")
for rank, idx in enumerate(ranked_dot_idx, 1):
    print(f"Rank {rank} (Score: {dot_scores[idx]:.4f}): {documents[idx]}")


=== SBERT Search Results ===

--- SBERT Cosine Similarity Rankings ---
Rank 1 (Score: 0.3832): A rich chocolate espresso cake garnished with strawberries.
Rank 2 (Score: 0.3769): A cozy Italian restaurant serving handmade pasta and pizza.
Rank 3 (Score: -0.0374): Central banks regulate interest rates and national inflation.
Rank 4 (Score: -0.0499): Commercial banks manage deposits, loans, and corporate accounts.

--- SBERT Dot Product Rankings ---
Rank 1 (Score: 0.3832): A rich chocolate espresso cake garnished with strawberries.
Rank 2 (Score: 0.3769): A cozy Italian restaurant serving handmade pasta and pizza.
Rank 3 (Score: -0.0374): Central banks regulate interest rates and national inflation.
Rank 4 (Score: -0.0499): Commercial banks manage deposits, loans, and corporate accounts.


In [15]:
# Now for GloVe model
embedder = TextEmbedder(glove_model)

doc_embeddings = embedder.encode(documents)
print(f"Corpus indexed! Matrix shape: {doc_embeddings.shape}")

query_vec = embedder.encode(user_query)

cos_scores = [cosine_sim(query_vec, doc_vec) for doc_vec in doc_embeddings]
dot_scores = [dot_sim(query_vec, doc_vec) for doc_vec in doc_embeddings]

# Step D: Rank ALL documents by relevance (highest to lowest score)
print("\n=== GloVe Search Results ===")
ranked_cos_idx = np.argsort(cos_scores)[::-1]
print("\n--- GloVe Cosine Similarity Rankings ---")
for rank, idx in enumerate(ranked_cos_idx, 1):
    print(f"Rank {rank} (Score: {cos_scores[idx]:.4f}): {documents[idx]}")

ranked_dot_idx = np.argsort(dot_scores)[::-1]
print("\n--- GloVe Dot Product Rankings ---")
for rank, idx in enumerate(ranked_dot_idx, 1):
    print(f"Rank {rank} (Score: {dot_scores[idx]:.4f}): {documents[idx]}")


Corpus indexed! Matrix shape: (4, 100)

=== GloVe Search Results ===

--- GloVe Cosine Similarity Rankings ---
Rank 1 (Score: 0.6830): A cozy Italian restaurant serving handmade pasta and pizza.
Rank 2 (Score: 0.6563): A rich chocolate espresso cake garnished with strawberries.
Rank 3 (Score: 0.6332): Commercial banks manage deposits, loans, and corporate accounts.
Rank 4 (Score: 0.6308): Central banks regulate interest rates and national inflation.

--- GloVe Dot Product Rankings ---
Rank 1 (Score: 14.1198): Central banks regulate interest rates and national inflation.
Rank 2 (Score: 13.9691): Commercial banks manage deposits, loans, and corporate accounts.
Rank 3 (Score: 12.5426): A rich chocolate espresso cake garnished with strawberries.
Rank 4 (Score: 12.2884): A cozy Italian restaurant serving handmade pasta and pizza.


## 9. Key Takeaways & Observations

* **SBERT (Cosine vs. Dot):**
  * Both metrics gave identical rankings (food docs at the top, banking docs at the bottom).
  * SBERT outputs normalized vectors (length = 1.0). When length is 1, `dot_product == cosine_similarity`.

* **GloVe (The Big Flaw):**
  * **Cosine:** Got it right (Italian restaurant at #1). It only compares the angle between vectors.
  * **Dot Product:** Got it completely wrong (Central banks at #1, Italian restaurant dead last at #4).
  * Why? Raw GloVe vectors are unnormalized. The banking sentence has words with huge vector lengths, and Dot Product multiplies angle by length. A massive vector length with a bad angle will beat a smaller vector with the right angle.

* **Rule of Thumb for Vector DBs:**
  * Always normalize embeddings first (unit length = 1.0).
  * Once normalized, Dot Product gives the exact same results as Cosine Similarity, but runs much faster in hardware.
